In [313]:
import kagglehub
import pandas as pd
import numpy as np
import json

In [314]:
path = kagglehub.dataset_download(
                            "tmdb/tmdb-movie-metadata",
                            output_dir = '../data')
movies_path =  path + "/tmdb_5000_movies.csv"
credits_path = path + "/tmdb_5000_credits.csv"

In [315]:
movies = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

In [316]:
def convert_to_hr_min(row):
    if row == 0:
        return ''
    else:
        hrs = int(row//60)
        mins = int(row - hrs * 60)
        return f'{hrs}h {mins}m'


In [317]:
movies.runtime = movies.runtime.fillna(0).apply(convert_to_hr_min).astype('string')
movies['release_year'] = movies.release_date.str[:4]

In [318]:
movies.runtime

0       2h 42m
1       2h 49m
2       2h 28m
3       2h 45m
4       2h 12m
         ...  
4798    1h 21m
4799    1h 25m
4800     2h 0m
4801    1h 38m
4802    1h 30m
Name: runtime, Length: 4803, dtype: string

In [319]:
movies_full = pd.merge(left = movies, right = credits, left_on = 'id', right_on = 'movie_id').reset_index(drop = True)

In [320]:
movies_df = movies_full[['title_x', 'tagline', 'release_year', 'overview', 'keywords', 'genres', 'cast', 'crew', 'runtime' ]]
movies_df = movies_df.rename({'title_x': 'title'}, axis = 1)

In [321]:
movies_df.head()

,title,tagline,release_year,overview,keywords,genres,cast,crew,runtime
0,Avatar,Enter the World of Pandora.,2009,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de...",2h 42m
1,Pirates of the Caribbean: At World's End,"At the end of the world, the adventure begins.",2007,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de...",2h 49m
2,Spectre,A Plan No One Escapes,2015,A cryptic message from Bond’s past sends him o...,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de...",2h 28m
3,The Dark Knight Rises,The Legend Ends,2012,Following the death of District Attorney Harve...,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...","[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[{""cast_id"": 2, ""character"": ""Bruce Wayne / Ba...","[{""credit_id"": ""52fe4781c3a36847f81398c3"", ""de...",2h 45m
4,John Carter,"Lost in our world, found in another.",2012,"John Carter is a war-weary, former military ca...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""cast_id"": 5, ""character"": ""John Carter"", ""c...","[{""credit_id"": ""52fe479ac3a36847f813eaa3"", ""de...",2h 12m


In [322]:
from ast import literal_eval

features = ['keywords', 'genres', 'cast', 'crew']

for feature in features:
    movies_df[feature] = movies_df[feature].apply(literal_eval)

In [323]:
movies_df.keywords

0       [{'id': 1463, 'name': 'culture clash'}, {'id':...
1       [{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...
2       [{'id': 470, 'name': 'spy'}, {'id': 818, 'name...
3       [{'id': 849, 'name': 'dc comics'}, {'id': 853,...
4       [{'id': 818, 'name': 'based on novel'}, {'id':...
                              ...                        
4798    [{'id': 5616, 'name': 'united states–mexico ba...
4799                                                   []
4800    [{'id': 248, 'name': 'date'}, {'id': 699, 'nam...
4801                                                   []
4802    [{'id': 1523, 'name': 'obsession'}, {'id': 224...
Name: keywords, Length: 4803, dtype: object

In [324]:
keyword_counts = {}
genre_counts = {}

for movie in movies_df['keywords']:
    for kw_data in movie:
        keyword = kw_data['name']
        keyword_counts[keyword] = keyword_counts.get(keyword, 0) + 1

for movie in movies_df['genres']:
    for genre_data in movie:
        genre = genre_data['name']
        genre_counts[genre] = genre_counts.get(genre, 0) + 1

In [325]:
movies_df.loc[0, 'keywords']

[{'id': 1463, 'name': 'culture clash'},
 {'id': 2964, 'name': 'future'},
 {'id': 3386, 'name': 'space war'},
 {'id': 3388, 'name': 'space colony'},
 {'id': 3679, 'name': 'society'},
 {'id': 3801, 'name': 'space travel'},
 {'id': 9685, 'name': 'futuristic'},
 {'id': 9840, 'name': 'romance'},
 {'id': 9882, 'name': 'space'},
 {'id': 9951, 'name': 'alien'},
 {'id': 10148, 'name': 'tribe'},
 {'id': 10158, 'name': 'alien planet'},
 {'id': 10987, 'name': 'cgi'},
 {'id': 11399, 'name': 'marine'},
 {'id': 13065, 'name': 'soldier'},
 {'id': 14643, 'name': 'battle'},
 {'id': 14720, 'name': 'love affair'},
 {'id': 165431, 'name': 'anti war'},
 {'id': 193554, 'name': 'power relations'},
 {'id': 206690, 'name': 'mind and soul'},
 {'id': 209714, 'name': '3d'}]

In [347]:
def top_keywords(row, length = 3):
    kw_counts = {kw['name'].replace(' ', ''): keyword_counts[kw['name']] for kw in row}
    kw_counts = sorted(kw_counts, key = lambda kw: kw_counts[kw], reverse = True)
    kw = [keyword for keyword in kw_counts]

    if len(kw) > length:
        return kw[:length]
    else:
        return kw
    
def top_genres(row, length, all = True):
    gen_counts = {genre['name'].replace(' ', ''): genre_counts[genre['name']] for genre in row}
    gen_counts = sorted(gen_counts, key = lambda gen: gen_counts[gen], reverse=True)
    genres = [genre for genre in gen_counts]

    if all:
        return genres
    
    if len(genres) > length:
        return genres[:length]
    else:
        return genres
    
    
def top_cast(row, length, format = True):
    if format:
        cast = [actor['name'].lower().replace(' ', '') for actor in row]
        cast = cast[:length]
    else:
        cast = [actor['name'] for actor in row]
        cast = cast[:length]
        cast = ', '.join(cast)
    
    return cast

def director(row, format = True):
    if format:
        director = [crew['name'].lower().replace(' ', '') for crew in row if crew['job'] == 'Director']
        return director
    else:
        director = [crew['name'] for crew in row if crew['job'] == 'Director']

        if director:
            return director[0]
        else:
            return ''

In [348]:
movies_df['top_keywords'] = movies_df['keywords'].apply(top_keywords, length = 6)
movies_df['top_cast'] = movies_df['cast'].apply(top_cast, length = 5)
movies_df['director'] = movies_df['crew'].apply(director)
movies_df['top_genres'] = movies_df['genres'].apply(top_genres, length = 3, all = True)
movies_df.rename({'title_x': 'title'}, axis = 1, inplace=True)

In [354]:
indices = movies_df[['title', 'overview', 'release_year', 'tagline', 'runtime', 'cast', 'crew']].reset_index().fillna('')
indices['top_cast'] = indices['cast'].apply(top_cast, length = 4, format = False)
indices['director'] = indices['crew'].apply(director, format = False)
indices = indices.set_index('title').drop(columns = ['cast', 'crew'])

indices.head()

,index,overview,release_year,tagline,runtime,top_cast,director
title,,,,,,,
Avatar,0,"In the 22nd century, a paraplegic Marine is di...",2009,Enter the World of Pandora.,2h 42m,"Sam Worthington, Zoe Saldana, Sigourney Weaver...",James Cameron
Pirates of the Caribbean: At World's End,1,"Captain Barbossa, long believed to be dead, ha...",2007,"At the end of the world, the adventure begins.",2h 49m,"Johnny Depp, Orlando Bloom, Keira Knightley, S...",Gore Verbinski
Spectre,2,A cryptic message from Bond’s past sends him o...,2015,A Plan No One Escapes,2h 28m,"Daniel Craig, Christoph Waltz, Léa Seydoux, Ra...",Sam Mendes
The Dark Knight Rises,3,Following the death of District Attorney Harve...,2012,The Legend Ends,2h 45m,"Christian Bale, Michael Caine, Gary Oldman, An...",Christopher Nolan
John Carter,4,"John Carter is a war-weary, former military ca...",2012,"Lost in our world, found in another.",2h 12m,"Taylor Kitsch, Lynn Collins, Samantha Morton, ...",Andrew Stanton


In [329]:
top_keywords_all = set()
for movie in movies_df['top_keywords']:
    top_keywords_all.update(movie)

top_keywords_cleaned = set()
for term in top_keywords_all:
    if ' ' in term:
        words = term.split(' ')
        top_keywords_cleaned.update(words)
    else:
        top_keywords_cleaned.add(term)

print(len(top_keywords_cleaned))

4801


In [330]:
top_genres_all = set()
for movie in movies_df['top_genres']:
    top_genres_all.update(movie)

top_genres_cleaned = set()
for term in top_genres_all:
    if ' ' in term:
        words = term.split(' ')
        top_genres_cleaned.update(words)
    else:
        top_genres_cleaned.add(term)

print(len(top_genres_cleaned))

20


In [331]:
set(movies_df['top_keywords'][0])

{'3d', 'alien', 'battle', 'future', 'soldier', 'space'}

In [332]:
movies_df.loc[(movies_df.title == 'Iron Man'), 'top_keywords'].to_list()

[['aftercreditsstinger',
  'superhero',
  'basedoncomicbook',
  'marvelcomic',
  'marvelcinematicuniverse',
  'middleeast']]

In [333]:
def create_soup(x):
    soup = x['top_keywords'] + x['top_cast'] + x['director'] + x['top_genres']
    return ' '.join(soup)

soup_df = pd.DataFrame(movies_df['title'])
soup_df['soup'] = movies_df.apply(create_soup, axis=1)
soup_df = soup_df.set_index('title')

In [334]:
soup_df.loc['Iron Man','soup']

'aftercreditsstinger superhero basedoncomicbook marvelcomic marvelcinematicuniverse middleeast robertdowneyjr. terrencehoward jeffbridges shauntoub gwynethpaltrow jonfavreau Action Adventure ScienceFiction'

In [335]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

count = CountVectorizer(stop_words='english')
count_matrix = normalize(count.fit_transform(soup_df['soup']), norm = 'l2')
cosine_sim_count = cosine_similarity(count_matrix, count_matrix)

In [336]:
count_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 63331 stored elements and shape (4803, 16846)>

In [337]:
soup_words_all = set()
for movie in soup_df['soup']:
    movie_kws = movie.split()
    soup_words_all.update(movie_kws)
print(len(soup_words_all))

16509


In [338]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(soup_df['soup'])
cosine_sim_tfidf = cosine_similarity(tfidf_matrix, tfidf_matrix)

In [339]:
dir(tfidf_matrix)
tfidf_matrix.indices

array([  14,  438, 5508, ..., 8067, 1955, 1928],
      shape=(63331,), dtype=int32)

In [340]:
def get_recommendations(title, cosine_sim):
    idx = indices.loc[title, 'index']
    sim_scores = enumerate(cosine_sim[idx])

    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)

    sim_scores = sim_scores[1:11]

    movie_indices = [i[0] for i in sim_scores]

    return movies_df['title'].iloc[movie_indices]

def get_recommendations_multi_input(movies, cosine_sim):
    sim_scores = []
    input_movie_indices = []
    for title in movies:
        idx = indices.loc[title].values[0]
        input_movie_indices.append(idx)
        sim_scores.append(cosine_sim[idx])

    weights = [3, 2, 1]

    sim_scores = np.average(np.array(sim_scores), axis = 0, weights = weights)
    
    sim_scores = enumerate(normalize(sim_scores.reshape(1,-1), norm='l2').flatten())

    sim_scores = sorted(sim_scores, key = lambda x: x[1], reverse = True)

    movie_indices = [i[0] for i in sim_scores if i[0] not in input_movie_indices][:10]

    # return movie_indices

    return indices.iloc[movie_indices].reset_index()['title']

In [341]:
movie = "12 Years a Slave"

display(
    # get_recommendations(movie, cosine_sim_count),
    get_recommendations(movie, cosine_sim_tfidf)
    )


3430                         Shame
188                           Salt
3750              The Great Escape
1051                     Prisoners
2522            The Imitation Game
979            Free State of Jones
912     Interview with the Vampire
877                     Black Mass
2507                     Slow Burn
1211                       Amistad
Name: title, dtype: object

In [342]:
movies_list = [
            'Housefull',
            'Babel',
            'Iron Man',
]

display(
# get_recommendations_multi_input(movies, cosine_sim_count),
get_recommendations_multi_input(movies_list, cosine_sim_tfidf)
)

0            Namastey London
1                 Iron Man 2
2      Me You and Five Bucks
3              Dum Maaro Dum
4              Amores perros
5    Yeh Jawaani Hai Deewani
6                 Iron Man 3
7                Khiladi 786
8         The Brothers Bloom
9                    Airlift
Name: title, dtype: object

In [ ]:
title_corrections = {
    'X2': 'X2: X-Men United',
    "Harry Potter and the Philosopher's Stone": "Harry Potter and the Sorcerer's Stone"
    ,'Fantastic 4: Rise of the Silver Surfer': 'Fantastic Four: Rise of the Silver Surfer'
    ,'The Taking of Pelham 1 2 3': 'The Taking of Pelham 123'
    ,'Miss Congeniality 2: Armed and Fabulous': 'Miss Congeniality 2: Armed & Fabulous'
}

indices.rename(index = title_corrections, inplace = True)

In [344]:
recos = get_recommendations_multi_input(movies_list, cosine_sim=cosine_sim_tfidf)

In [358]:
indices

,index,overview,release_year,tagline,runtime,top_cast,director
title,,,,,,,
Avatar,0,"In the 22nd century, a paraplegic Marine is di...",2009,Enter the World of Pandora.,2h 42m,"Sam Worthington, Zoe Saldana, Sigourney Weaver...",James Cameron
Pirates of the Caribbean: At World's End,1,"Captain Barbossa, long believed to be dead, ha...",2007,"At the end of the world, the adventure begins.",2h 49m,"Johnny Depp, Orlando Bloom, Keira Knightley, S...",Gore Verbinski
Spectre,2,A cryptic message from Bond’s past sends him o...,2015,A Plan No One Escapes,2h 28m,"Daniel Craig, Christoph Waltz, Léa Seydoux, Ra...",Sam Mendes
The Dark Knight Rises,3,Following the death of District Attorney Harve...,2012,The Legend Ends,2h 45m,"Christian Bale, Michael Caine, Gary Oldman, An...",Christopher Nolan
John Carter,4,"John Carter is a war-weary, former military ca...",2012,"Lost in our world, found in another.",2h 12m,"Taylor Kitsch, Lynn Collins, Samantha Morton, ...",Andrew Stanton
...,...,...,...,...,...,...,...
El Mariachi,4798,El Mariachi just wants to play his guitar and ...,1992,"He didn't come looking for trouble, but troubl...",1h 21m,"Carlos Gallardo, Jaime de Hoyos, Peter Marquar...",Robert Rodriguez
Newlyweds,4799,A newlywed couple's honeymoon is upended by th...,2011,A newlywed couple's honeymoon is upended by th...,1h 25m,"Edward Burns, Kerry Bishé, Marsha Dietlein, Ca...",Edward Burns
"Signed, Sealed, Delivered",4800,"""Signed, Sealed, Delivered"" introduces a dedic...",2013,,2h 0m,"Eric Mabius, Kristin Booth, Crystal Lowe, Geof...",Scott Smith


In [365]:
title_corrections = {}

with open('correct_titles.csv', 'r') as f:
    rows = f.readlines()
    
for row in rows:
    inc_title, cor_title = row.split('\",\"')
    inc_title = inc_title[1:]
    cor_title = cor_title[:-2]
    title_corrections[inc_title] = cor_title


In [367]:
indices = indices.rename(title_corrections)

In [369]:
import joblib
import os

os.makedirs('../artifacts', exist_ok=True)

cosine_sim_path = '../artifacts/cosine_sim.pkl'
indices_path = '../artifacts/indices.pkl'

joblib.dump(cosine_sim_tfidf.astype('float32'), cosine_sim_path)
joblib.dump(indices, indices_path)

['../artifacts/indices.pkl']